# بايبلاين صور الأداء (Performers) — تحميل + فحص وجوه + رفع Parquet على Hugging Face Bucket

النوتبوك ده بيشغّل البايبلاين المقسّم على **10 دفعات حسب المستخدمين/السجلات في ملف الـJSON، وليس حسب عدد الصور**.

كل مستخدم يدخل في دفعة واحدة فقط، وكل `image_urls` الخاصة به تُحمّل وتُفحص داخل نفس الدفعة حتى لو كان لديه عدد كبير من الصور.

لكل دفعة:

1. تحديد مجموعة المستخدمين الخاصة بالدفعة من ملف الـJSON.
2. تحميل كل الصور الموجودة في `image_urls` لهؤلاء المستخدمين.
3. تخزين صور كل مستخدم داخل مجلد باسم الـ`id` الخاص به.
4. فحص الصور بموديل YOLOv11l-face وكتابة التقرير.
5. تحويل الصور الأصلية إلى ملف Parquet واحد للدفعة **بدون تشفير**.
6. رفع ملف Parquet + تقرير فحص الوجوه إلى Hugging Face Bucket.
7. التأكد من الرفع على السيرفر.
8. تحديث التقرير الرئيسي (master report) لاستخدامه عند الاستئناف.
9. حذف ملفات الدفعة محلياً والانتقال للدفعة التالية.

> ⚠️ لازم تفعّل **GPU** و **Internet** من (Settings) في النوتبوك قبل التشغيل.

## مساحة التخزين: `/kaggle/temp` مش `/kaggle/working`

البيانات كبيرة، لذلك كل التحميل والملفات المؤقتة للدفعة بتتحط في `/kaggle/temp` وبتتمسح بعد نجاح الرفع والتحقق، بحيث لا نحتاج الاحتفاظ بكل البيانات محلياً في نفس الوقت.


## 1) الإعدادات

املأ القيم دي قبل التشغيل. المفتاح السري الوحيد المطلوب هو `HF_TOKEN`.


In [ ]:
# ==== إعدادات المشروع ====
REPO_URL = "https://github.com/kareemkamal10/app.git"

# رابط ملف الـ JSON بتاع الأداء (المصدر)
JSON_SOURCE = "https://huggingface.co/datasets/abdelwahabnabil500/datafile/resolve/main/stashdb_performers_full.json"

# اسم الـ Hugging Face Bucket اللي هيترفع عليه كل حاجة
BUCKET = "abdelwahabnabil500/faces"

# عدد دفعات المستخدمين، وعدد الـ workers للتحميل المتوازي، وحجم mini-batch لموديل الوجوه
BATCH_COUNT = 10
WORKERS = 16
DETECT_BATCH_SIZE = 64
DEVICES = "0,1"  # الكارتين على Kaggle (T4 x2)

# ==== المفتاح السري الوحيد ====
# الأفضل تسجيله في Kaggle Secrets باسم HF_TOKEN
HF_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        _val = _secrets.get_secret("HF_TOKEN")
    except Exception:
        _val = None
    if _val:
        HF_TOKEN = _val
        print("تم قراءة HF_TOKEN من Kaggle Secrets.")
except Exception:
    pass  # Kaggle Secrets مش متاحة، هنستخدم القيمة المكتوبة فوق

assert REPO_URL, "REPO_URL لازم يكون متملي."
assert BUCKET, "BUCKET لازم يكون متملي."
assert JSON_SOURCE, "JSON_SOURCE لازم يكون متملي."
assert HF_TOKEN, "HF_TOKEN فاضي — سجّله في Kaggle Secrets أو اكتبه فوق."


## 2) فحص الـ GPU

In [ ]:
!nvidia-smi

## 3) استنساخ المشروع من GitHub

In [ ]:
import os, shutil

PROJECT_DIR = "/kaggle/working/project"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

!git clone "{REPO_URL}" "{PROJECT_DIR}"


In [ ]:
%cd {PROJECT_DIR}
!ls -la

## 4) تثبيت المتطلبات

In [ ]:
!pip install -q -r requirements.txt

## 5) اختبار سريع (اختياري) — 100 صورة بس

الخلية دي بتجرّب مسار التحميل + فحص الوجوه + إنشاء Parquet عادي + الرفع والتحقق على 100 صورة فقط، عشان تتأكد إن المكونات الأساسية شغالة قبل ما تبدأ الدفعات الحقيقية.

الاختبار يستخدم `batch_00` كمساحة اختبار منفصلة، ولا يدخل في حالة الاستئناف للدفعات الحقيقية 1-10.


In [ ]:
import os

os.environ["HF_TOKEN"] = HF_TOKEN
!python 00_smoke_test.py \
    --json-source "{JSON_SOURCE}" \
    --bucket "{BUCKET}" \
    --image-count 100 \
    --devices "{DEVICES}" \
    --batch-size {DETECT_BATCH_SIZE} \
    --work-dir /kaggle/temp/pipeline_smoke_test


## 6) تشغيل البايبلاين الكامل (10 دفعات مستخدمين)

الخلية دي بتشغّل `batch_pipeline.py` بمساره الكامل، وبتستخدم `/kaggle/temp` كمساحة عمل مؤقتة.

التقسيم الحقيقي يتم حسب سجلات المستخدمين في الـJSON: كل مستخدم وجميع `image_urls` الخاصة به في نفس الدفعة.

لو الجلسة اتقطعت وشغّلتها تاني، هتقرأ آخر تقرير رئيسي من الـBucket وتكمل من الدفعة التي لم تكتمل.


In [ ]:
import os, subprocess, sys

os.environ["HF_TOKEN"] = HF_TOKEN

WORK_DIR = "/kaggle/temp/pipeline_work"  # مش /kaggle/working -- عشان المساحة (50 جيجا بدل 20)

cmd = [
    sys.executable, "batch_pipeline.py",
    "--json-source", JSON_SOURCE,
    "--bucket", BUCKET,
    "--batch-count", str(BATCH_COUNT),
    "--workers", str(WORKERS),
    "--devices", DEVICES,
    "--batch-size", str(DETECT_BATCH_SIZE),
    "--work-dir", WORK_DIR,
]

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"البايبلاين فشل بكود خروج {process.returncode} -- شغّل الخلية تاني وهيكمل من آخر دفعة اترفعت.")
print("\nخلص البايبلاين بنجاح.")


## 7) عرض التقرير الرئيسي النهائي

In [ ]:
import json
from pathlib import Path

master_path = Path(WORK_DIR) / "reports" / "download_master_report.md"
if master_path.exists():
    print(master_path.read_text(encoding="utf-8"))
else:
    print("مفيش تقرير رئيسي محلي -- شوف نسخته على الـ Bucket في final/download_master_report.md")
